# GPT-2 Inference Benchmark
Measures tokens/sec, time to first token, and peak GPU memory for GPT-2 small on a T4,
comparing fp32 vs fp16 and KV cache on vs off at batch sizes 1, 8, and 32.

In [ ]:
import torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained("gpt2")
tok.pad_token = tok.eos_token
tok.padding_side = "left"

In [ ]:
def bench(model, batch_size, use_cache=True, prompt="The future of physics is", n_tokens=300, runs=10, warmup=10):
    inputs = tok([prompt] * batch_size, return_tensors="pt", padding=True).to("cuda")
    kw = dict(do_sample=False, pad_token_id=tok.eos_token_id, use_cache=use_cache)
    full = dict(max_new_tokens=n_tokens, min_new_tokens=n_tokens, **kw)

    for i in range(warmup):  #warm-up
        model.generate(**inputs, **full)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    t = time.time()  #ttft
    for i in range(runs):
        model.generate(**inputs, max_new_tokens=1, **kw)
    torch.cuda.synchronize()
    ttft_ms = (time.time() - t) / runs * 1000

    t = time.time()  #tokens/second
    for i in range(runs):
        model.generate(**inputs, **full)
    torch.cuda.synchronize()
    tps = batch_size * n_tokens * runs / (time.time() - t)

    return ttft_ms, tps, torch.cuda.max_memory_allocated() / 1e6  #VRAM

In [ ]:
configs = [
    ("fp32, cache on",  torch.float32, True),
    ("fp32, cache off", torch.float32, False),
    ("fp16, cache on",  torch.float16, True),
    ("fp16, cache off", torch.float16, False),
]

print("| config | batch | TTFT (ms) | tokens/sec | peak mem (MB) |")
print("|---|---|---|---|---|")
for name, dtype, cache in configs:
    model = AutoModelForCausalLM.from_pretrained("gpt2", dtype=dtype).to("cuda")
    for bs in [1, 8, 32]:
        ttft, tps, mem = bench(model, bs, use_cache=cache)
        print(f"| {name} | {bs} | {ttft:.1f} | {tps:.0f} | {mem:.0f} |")
    del model
    torch.cuda.empty_cache()

## Perplexity
Checks that fp16 does not hurt quality. WikiText-2 test set non-overlapping 1024-token chunks (lower is better).

In [ ]:
import math
from datasets import load_dataset

test = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
enc = tok("\n\n".join(test["text"]), return_tensors="pt").input_ids

@torch.no_grad()
def perplexity(model, max_len=1024):
    nll, count = 0.0, 0
    for i in range(0, enc.size(1) - max_len, max_len):
        ids = enc[:, i:i + max_len].to("cuda")
        loss = model(ids, labels=ids).loss
        nll += loss.float().item() * (max_len - 1)
        count += max_len - 1
    return math.exp(nll / count)

for name, dtype in [("fp32", torch.float32), ("fp16", torch.float16)]:
    m = AutoModelForCausalLM.from_pretrained("gpt2", dtype=dtype).to("cuda")
    print(name, "perplexity:", round(perplexity(m), 2))
    del m
    torch.cuda.empty_cache()

## torch.compile
Uses a static KV cache (which makes Hugging Face compile the forward pass automatically) to cut per-step CPU overhead. I skipped the CUDA graphs (the in-place cache update conflicts with them), so the speedup comes from kernel fusion.

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gpt2", dtype=torch.float16).to("cuda")
model.generation_config.cache_implementation = "static"

for bs in [1, 8, 32]:
    ttft, tps, mem = bench(model, bs)
    print(f"| fp16, static cache, auto-compiled | {bs} | {ttft:.1f} | {tps:.0f} | {mem:.0f} |")

## Correctness Check

In [ ]:
ref = AutoModelForCausalLM.from_pretrained("gpt2", dtype=torch.float16).to("cuda")
inp = tok("The future of physics is", return_tensors="pt").to("cuda")
kw = dict(max_new_tokens=50, do_sample=False, pad_token_id=tok.eos_token_id)

a = ref.generate(**inp, **kw)
b = model.generate(**inp, **kw)
print("identical output:", torch.equal(a, b))

import gc
del ref, model
gc.collect()
torch.cuda.empty_cache()

## Long prompt (prefill)

In [ ]:
long_prompt = "The future of physics is " * 100   # approx 500 tokens

print("| config | prompt | TTFT (ms) | tokens/sec | peak mem (MB) |")
print("|---|---|---|---|---|")
for name, dtype in [("fp32", torch.float32), ("fp16", torch.float16)]:
    m = AutoModelForCausalLM.from_pretrained("gpt2", dtype=dtype).to("cuda")
    for bs in [1, 8]:
        ttft, tps, mem = bench(m, bs, prompt=long_prompt, n_tokens=100)
        print(f"| {name}, batch {bs} | ~500 tok | {ttft:.1f} | {tps:.0f} | {mem:.0f} |")
    del m
    torch.cuda.empty_cache()

## Profiling

In [ ]:
import time
from torch.profiler import profile, ProfilerActivity

m = AutoModelForCausalLM.from_pretrained("gpt2", dtype=torch.float16).to("cuda")
inputs = tok(["The future of physics is"], return_tensors="pt", padding=True).to("cuda")
kw = dict(max_new_tokens=50, min_new_tokens=50, do_sample=False, pad_token_id=tok.eos_token_id)

for _ in range(5):          # warmup
    m.generate(**inputs, **kw)
torch.cuda.synchronize()

t = time.time()
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    m.generate(**inputs, **kw)
    torch.cuda.synchronize()
wall_ms = (time.time() - t) * 1000

print(f"wall time for 50 tokens: {wall_ms:.0f} ms")
print(prof.key_averages().table(sort_by="self_device_time_total", row_limit=10))

del m
gc.collect()
torch.cuda.empty_cache()

## INT8 quantization

In [ ]:
!pip install -q bitsandbytes

import logging
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)

from transformers import BitsAndBytesConfig

cfg = BitsAndBytesConfig(load_in_8bit=True)
m8 = AutoModelForCausalLM.from_pretrained(
    "gpt2", quantization_config=cfg, device_map="cuda", dtype=torch.float16
)

print("| config | batch | TTFT (ms) | tokens/sec | peak mem (MB) |")
print("|---|---|---|---|---|")
for bs in [1, 8]:
    ttft, tps, mem = bench(m8, bs, n_tokens=100, runs=3, warmup=1)
    print(f"| int8 (bitsandbytes), cache on | {bs} | {ttft:.1f} | {tps:.0f} | {mem:.0f} |")

print("int8 perplexity:", round(perplexity(m8), 2))